<a href="https://www.kaggle.com/code/muhammaddhiyaulatha/arc-baseline-zero-model-ipynb?scriptVersionId=313476324" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ARC Prize 2026 - Baseline Model

This notebook contains my first submission to the ARC-AGI-2 competition on Kaggle.

## Approach

* Generate output grids filled with zeros
* Match input grid dimensions
* Ensure correct submission format

## Purpose

This is a baseline to understand:

* Submission pipeline
* Evaluation system
* Dataset structure

Next step: implement rule-based reasoning.


In [1]:
import json
import os
from collections import Counter, defaultdict
from itertools import product

# ============================================================================
# 0. SETUP
# ============================================================================
is_rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

if is_rerun:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
else:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

with open(path) as f:
    data = json.load(f)

# ============================================================================
# 1. GRID UTILITIES
# ============================================================================
def copy_grid(g):
    return [row[:] for row in g]

def grid_dims(g):
    return (len(g), len(g[0])) if g and g[0] else (0, 0)

def flatten(g):
    return [c for row in g for c in row]

def grids_equal(a, b):
    if len(a) != len(b):
        return False
    return all(ra == rb for ra, rb in zip(a, b))

def unique_colors(g):
    return set(flatten(g))

def most_common_color(grid):
    return Counter(flatten(grid)).most_common(1)[0][0]

def background_color(grid):
    return most_common_color(grid)

def color_counts(grid):
    return Counter(flatten(grid))

def grid_to_tuple(g):
    return tuple(tuple(r) for r in g)

# ============================================================================
# 2. GEOMETRIC TRANSFORMS
# ============================================================================
def rotate90(g):
    return list(map(list, zip(*g[::-1])))

def rotate180(g):
    return rotate90(rotate90(g))

def rotate270(g):
    return rotate90(rotate90(rotate90(g)))

def flip_h(g):
    return [row[::-1] for row in g]

def flip_v(g):
    return g[::-1]

def transpose(g):
    h, w = grid_dims(g)
    return [[g[i][j] for i in range(h)] for j in range(w)]

def transpose_anti(g):
    return rotate90(flip_v(g))

ALL_RIGID_TRANSFORMS = [
    ("identity", lambda x: copy_grid(x)),
    ("rot90", rotate90),
    ("rot180", rotate180),
    ("rot270", rotate270),
    ("flip_h", flip_h),
    ("flip_v", flip_v),
    ("transpose", transpose),
    ("transpose_anti", transpose_anti),
]

# ============================================================================
# 3. COLOR MAPPING
# ============================================================================
def color_map(g, mapping):
    return [[mapping.get(c, c) for c in row] for row in g]

def infer_color_map(train):
    mapping = {}
    
    for pair in train:
        inp, out = pair["input"], pair["output"]
        
        h = min(len(inp), len(out))
        w = min(len(inp[0]), len(out[0]))
        
        for i in range(h):
            for j in range(w):
                mapping[inp[i][j]] = out[i][j]
    
    return mapping

# ============================================================================
# 4. CROPPING & BOUNDING BOX
# ============================================================================
def bounding_box(grid, ignore_color=None):
    h, w = grid_dims(grid)
    if ignore_color is None:
        ignore_color = background_color(grid)
    min_r, max_r, min_c, max_c = h, -1, w, -1
    for i in range(h):
        for j in range(w):
            if grid[i][j] != ignore_color:
                min_r = min(min_r, i)
                max_r = max(max_r, i)
                min_c = min(min_c, j)
                max_c = max(max_c, j)
    if max_r == -1:
        return 0, 0, h, w
    return min_r, min_c, max_r + 1, max_c + 1

def crop(grid, r1, c1, r2, c2):
    return [row[c1:c2] for row in grid[r1:r2]]

def crop_to_content(grid, ignore_color=None):
    r1, c1, r2, c2 = bounding_box(grid, ignore_color)
    return crop(grid, r1, c1, r2, c2)

def crop_color(grid, color):
    h, w = grid_dims(grid)
    min_r, max_r, min_c, max_c = h, -1, w, -1
    for i in range(h):
        for j in range(w):
            if grid[i][j] == color:
                min_r = min(min_r, i)
                max_r = max(max_r, i)
                min_c = min(min_c, j)
                max_c = max(max_c, j)
    if max_r == -1:
        return [[]]
    return crop(grid, min_r, min_c, max_r + 1, max_c + 1)

# ============================================================================
# 5. TILING & SCALING
# ============================================================================
def tile_grid(g, reps_h, reps_w):
    result = []
    for _ in range(reps_h):
        for row in g:
            new_row = []
            for _ in range(reps_w):
                new_row.extend(row)
            result.append(new_row)
    return result

def scale_grid(g, factor_h, factor_w):
    result = []
    for row in g:
        new_row = []
        for c in row:
            new_row.extend([c] * factor_w)
        for _ in range(factor_h):
            result.append(new_row[:])
    return result

def downscale_grid(g, factor_h, factor_w):
    h, w = grid_dims(g)
    nh, nw = h // factor_h, w // factor_w
    result = []
    for i in range(nh):
        row = []
        for j in range(nw):
            block = []
            for di in range(factor_h):
                for dj in range(factor_w):
                    block.append(g[i * factor_h + di][j * factor_w + dj])
            row.append(Counter(block).most_common(1)[0][0])
        result.append(row)
    return result

# ============================================================================
# 6. FLOOD FILL & CONNECTED COMPONENTS
# ============================================================================
def flood_fill_positions(grid, start_r, start_c, color=None, visited=None):
    h, w = grid_dims(grid)
    if color is None:
        color = grid[start_r][start_c]
    if visited is None:
        visited = set()
    stack = [(start_r, start_c)]
    positions = []
    while stack:
        r, c = stack.pop()
        if (r, c) in visited or r < 0 or r >= h or c < 0 or c >= w:
            continue
        if grid[r][c] != color:
            continue
        visited.add((r, c))
        positions.append((r, c))
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            stack.append((r + dr, c + dc))
    return positions

def connected_components(grid, ignore_color=None):
    h, w = grid_dims(grid)
    visited = set()
    components = []
    for i in range(h):
        for j in range(w):
            if (i, j) not in visited and (ignore_color is None or grid[i][j] != ignore_color):
                cc = flood_fill_positions(grid, i, j, visited=visited)
                components.append((grid[i][j], cc))
    return components

def extract_objects(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    comps = connected_components(grid, ignore_color=bg)
    objects = []
    for color, positions in comps:
        min_r = min(r for r, c in positions)
        max_r = max(r for r, c in positions)
        min_c = min(c for r, c in positions)
        max_c = max(c for r, c in positions)
        oh = max_r - min_r + 1
        ow = max_c - min_c + 1
        obj_grid = [[bg] * ow for _ in range(oh)]
        for r, c in positions:
            obj_grid[r - min_r][c - min_c] = color
        objects.append({
            "color": color,
            "positions": positions,
            "grid": obj_grid,
            "bbox": (min_r, min_c, max_r + 1, max_c + 1),
            "size": len(positions)
        })
    return objects

def extract_multicolor_objects(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    visited = set()
    objects = []
    for i in range(h):
        for j in range(w):
            if (i, j) not in visited and grid[i][j] != bg:
                stack = [(i, j)]
                positions = []
                while stack:
                    r, c = stack.pop()
                    if (r, c) in visited or r < 0 or r >= h or c < 0 or c >= w:
                        continue
                    if grid[r][c] == bg:
                        continue
                    visited.add((r, c))
                    positions.append((r, c))
                    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                        stack.append((r + dr, c + dc))
                if positions:
                    min_r = min(r for r, c in positions)
                    max_r = max(r for r, c in positions)
                    min_c = min(c for r, c in positions)
                    max_c = max(c for r, c in positions)
                    oh = max_r - min_r + 1
                    ow = max_c - min_c + 1
                    obj_grid = [[bg] * ow for _ in range(oh)]
                    for r, c in positions:
                        obj_grid[r - min_r][c - min_c] = grid[r][c]
                    objects.append({
                        "positions": positions,
                        "grid": obj_grid,
                        "bbox": (min_r, min_c, max_r + 1, max_c + 1),
                        "size": len(positions)
                    })
    return objects

# ============================================================================
# 7. SYMMETRY
# ============================================================================
def make_horizontally_symmetric(g):
    result = copy_grid(g)
    h, w = grid_dims(g)
    bg = background_color(g)
    for i in range(h):
        for j in range(w // 2):
            j2 = w - 1 - j
            if result[i][j] != bg and result[i][j2] == bg:
                result[i][j2] = result[i][j]
            elif result[i][j2] != bg and result[i][j] == bg:
                result[i][j] = result[i][j2]
    return result

def make_vertically_symmetric(g):
    result = copy_grid(g)
    h, w = grid_dims(g)
    bg = background_color(g)
    for i in range(h // 2):
        i2 = h - 1 - i
        for j in range(w):
            if result[i][j] != bg and result[i2][j] == bg:
                result[i2][j] = result[i][j]
            elif result[i2][j] != bg and result[i][j] == bg:
                result[i][j] = result[i2][j]
    return result

def make_4way_symmetric(g):
    return make_vertically_symmetric(make_horizontally_symmetric(g))

def complete_symmetry_bg_aware(grid):
    bg = background_color(grid)
    h, w = grid_dims(grid)
    result = copy_grid(grid)
    for i in range(h):
        for j in range(w):
            j2 = w - 1 - j
            if result[i][j] != bg and result[i][j2] == bg:
                result[i][j2] = result[i][j]
    for i in range(h):
        i2 = h - 1 - i
        for j in range(w):
            if result[i][j] != bg and result[i2][j] == bg:
                result[i2][j] = result[i][j]
    return result

# ============================================================================
# 8. BORDER OPERATIONS
# ============================================================================
def extract_border(grid):
    h, w = grid_dims(grid)
    border = []
    for j in range(w):
        border.append(grid[0][j])
        if h > 1:
            border.append(grid[h - 1][j])
    for i in range(1, h - 1):
        border.append(grid[i][0])
        if w > 1:
            border.append(grid[i][w - 1])
    return border

def remove_border(grid):
    if len(grid) <= 2 or len(grid[0]) <= 2:
        return copy_grid(grid)
    return [row[1:-1] for row in grid[1:-1]]

def add_border(grid, color):
    h, w = grid_dims(grid)
    new_w = w + 2
    result = [[color] * new_w]
    for row in grid:
        result.append([color] + row[:] + [color])
    result.append([color] * new_w)
    return result

# ============================================================================
# 9. GRAVITY
# ============================================================================
def gravity_down(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    result = [[bg] * w for _ in range(h)]
    for j in range(w):
        col_vals = [grid[i][j] for i in range(h) if grid[i][j] != bg]
        for idx, val in enumerate(reversed(col_vals)):
            result[h - 1 - idx][j] = val
    return result

def gravity_up(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    result = [[bg] * w for _ in range(h)]
    for j in range(w):
        col_vals = [grid[i][j] for i in range(h) if grid[i][j] != bg]
        for idx, val in enumerate(col_vals):
            result[idx][j] = val
    return result

def gravity_left(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        row_vals = [grid[i][j] for j in range(w) if grid[i][j] != bg]
        for idx, val in enumerate(row_vals):
            result[i][idx] = val
    return result

def gravity_right(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        row_vals = [grid[i][j] for j in range(w) if grid[i][j] != bg]
        for idx, val in enumerate(reversed(row_vals)):
            result[i][w - 1 - idx] = val
    return result

# ============================================================================
# 10. GRID BOOLEAN OPERATIONS
# ============================================================================
def or_grids(g1, g2, bg=0):
    h = min(len(g1), len(g2))
    w = min(len(g1[0]), len(g2[0]))
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        for j in range(w):
            if g1[i][j] != bg:
                result[i][j] = g1[i][j]
            elif g2[i][j] != bg:
                result[i][j] = g2[i][j]
    return result

def and_grids(g1, g2, bg=0):
    h = min(len(g1), len(g2))
    w = min(len(g1[0]), len(g2[0]))
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        for j in range(w):
            if g1[i][j] == g2[i][j]:
                result[i][j] = g1[i][j]
    return result

def xor_grids(g1, g2, bg=0):
    h = min(len(g1), len(g2))
    w = min(len(g1[0]), len(g2[0]))
    result = [[bg] * w for _ in range(h)]
    for i in range(h):
        for j in range(w):
            if g1[i][j] != g2[i][j]:
                result[i][j] = g2[i][j]
    return result

# ============================================================================
# 11. GRID SPLITTING & CONCAT
# ============================================================================
def split_grid_horizontal(g, n):
    h, w = grid_dims(g)
    ph = h // n
    return [[row[:] for row in g[i * ph:(i + 1) * ph]] for i in range(n)]

def split_grid_vertical(g, n):
    h, w = grid_dims(g)
    pw = w // n
    return [[row[i * pw:(i + 1) * pw] for row in g] for i in range(n)]

def concat_horizontal(grids):
    if not grids:
        return [[]]
    h = len(grids[0])
    result = []
    for i in range(h):
        row = []
        for g in grids:
            if i < len(g):
                row.extend(g[i])
        result.append(row)
    return result

def concat_vertical(grids):
    result = []
    for g in grids:
        result.extend([row[:] for row in g])
    return result

# ============================================================================
# 12. REPEATING PATTERN & DIVIDERS
# ============================================================================
def find_repeating_unit(grid):
    h, w = grid_dims(grid)
    for uh in range(1, h + 1):
        if h % uh != 0:
            continue
        for uw in range(1, w + 1):
            if w % uw != 0:
                continue
            unit = [row[:uw] for row in grid[:uh]]
            if grids_equal(tile_grid(unit, h // uh, w // uw), grid):
                if uh < h or uw < w:
                    return unit
    return grid

def detect_grid_divisions(grid):
    h, w = grid_dims(grid)
    h_dividers = []
    for i in range(h):
        if len(set(grid[i])) == 1:
            h_dividers.append(i)
    v_dividers = []
    for j in range(w):
        col = [grid[i][j] for i in range(h)]
        if len(set(col)) == 1:
            v_dividers.append(j)
    return h_dividers, v_dividers

def split_by_dividers(grid, h_dividers, v_dividers):
    h, w = grid_dims(grid)
    def make_ranges(dividers, total):
        ranges = []
        prev = 0
        for d in sorted(dividers):
            if d > prev:
                ranges.append((prev, d))
            prev = d + 1
        if prev < total:
            ranges.append((prev, total))
        return ranges

    h_ranges = make_ranges(h_dividers, h)
    v_ranges = make_ranges(v_dividers, w)
    cells = []
    for r1, r2 in h_ranges:
        row_cells = []
        for c1, c2 in v_ranges:
            row_cells.append(crop(grid, r1, c1, r2, c2))
        cells.append(row_cells)
    return cells

# ============================================================================
# 13. FILL OPERATIONS
# ============================================================================
def fill_enclosed_regions(grid, bg=None):
    if bg is None:
        bg = background_color(grid)
    h, w = grid_dims(grid)
    result = copy_grid(grid)
    border_reachable = set()
    stack = []
    for i in range(h):
        if grid[i][0] == bg:
            stack.append((i, 0))
        if w > 1 and grid[i][w - 1] == bg:
            stack.append((i, w - 1))
    for j in range(w):
        if grid[0][j] == bg:
            stack.append((0, j))
        if h > 1 and grid[h - 1][j] == bg:
            stack.append((h - 1, j))
    while stack:
        r, c = stack.pop()
        if (r, c) in border_reachable:
            continue
        if r < 0 or r >= h or c < 0 or c >= w:
            continue
        if grid[r][c] != bg:
            continue
        border_reachable.add((r, c))
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            stack.append((r + dr, c + dc))
    for i in range(h):
        for j in range(w):
            if grid[i][j] == bg and (i, j) not in border_reachable:
                neighbors = []
                for di in range(-2, 3):
                    for dj in range(-2, 3):
                        ni, nj = i + di, j + dj
                        if 0 <= ni < h and 0 <= nj < w and grid[ni][nj] != bg:
                            neighbors.append(grid[ni][nj])
                if neighbors:
                    result[i][j] = Counter(neighbors).most_common(1)[0][0]
    return result

def replace_color(grid, old_color, new_color):
    return [[new_color if c == old_color else c for c in row] for row in grid]

# ============================================================================
# 14. ROW/COL OPERATIONS
# ============================================================================
def duplicate_rows(grid, n):
    result = []
    for row in grid:
        for _ in range(n):
            result.append(row[:])
    return result

def duplicate_cols(grid, n):
    return transpose(duplicate_rows(transpose(grid), n))

# ============================================================================
# 15. ALL STRATEGY FUNCTIONS
# ============================================================================
def try_output_constant(train):
    if len(train) < 2:
        return None
    out0 = train[0]["output"]
    if all(grids_equal(p["output"], out0) for p in train[1:]):
        return lambda inp, o=copy_grid(out0): copy_grid(o)
    return None

def try_rigid_transforms(train):
    for name, op in ALL_RIGID_TRANSFORMS:
        if all(grids_equal(op(p["input"]), p["output"]) for p in train):
            return op
    return None

def try_color_map_only(train):
    cmap = infer_color_map(train)
    identity = all(cmap.get(k, k) == k for k in cmap)
    if identity:
        return None
    if all(grids_equal(color_map(p["input"], cmap), p["output"]) for p in train):
        return lambda inp, m=cmap: color_map(inp, m)
    return None

def try_transform_then_color_map(train):
    for name, op in ALL_RIGID_TRANSFORMS:
        votes = defaultdict(list)
        ok = True
        for p in train:
            t = op(p["input"])
            out = p["output"]
            if grid_dims(t) != grid_dims(out):
                ok = False
                break
            h, w = grid_dims(t)
            for i in range(h):
                for j in range(w):
                    votes[t[i][j]].append(out[i][j])
        if not ok:
            continue
        cmap = {k: Counter(v).most_common(1)[0][0] for k, v in votes.items()}
        if all(grids_equal(color_map(op(p["input"]), cmap), p["output"]) for p in train):
            return lambda inp, o=op, m=cmap: color_map(o(inp), m)
    return None

def try_crop_to_content_strategy(train):
    for p in train:
        bg = background_color(p["input"])
        cropped = crop_to_content(p["input"], bg)
        if not grids_equal(cropped, p["output"]):
            return None
    return lambda inp: crop_to_content(inp, background_color(inp))

def try_crop_to_specific_color(train):
    all_colors = set()
    for p in train:
        all_colors |= unique_colors(p["input"])
    for c in all_colors:
        if all(grids_equal(crop_color(p["input"], c), p["output"]) for p in train):
            return lambda inp, col=c: crop_color(inp, col)
    return None

def try_scale(train):
    for fh in range(1, 8):
        for fw in range(1, 8):
            if fh == 1 and fw == 1:
                continue
            if all(
                len(p["output"]) == len(p["input"]) * fh and
                len(p["output"][0]) == len(p["input"][0]) * fw and
                grids_equal(scale_grid(p["input"], fh, fw), p["output"])
                for p in train
            ):
                return lambda inp, a=fh, b=fw: scale_grid(inp, a, b)
    return None

def try_tile(train):
    for rh in range(1, 8):
        for rw in range(1, 8):
            if rh == 1 and rw == 1:
                continue
            if all(
                len(p["output"]) == len(p["input"]) * rh and
                len(p["output"][0]) == len(p["input"][0]) * rw and
                grids_equal(tile_grid(p["input"], rh, rw), p["output"])
                for p in train
            ):
                return lambda inp, a=rh, b=rw: tile_grid(inp, a, b)
    return None

def try_downscale(train):
    for fh in range(2, 6):
        for fw in range(2, 6):
            ok = True
            for p in train:
                ih, iw = grid_dims(p["input"])
                oh, ow = grid_dims(p["output"])
                if ih % fh != 0 or iw % fw != 0 or ih // fh != oh or iw // fw != ow:
                    ok = False
                    break
                if not grids_equal(downscale_grid(p["input"], fh, fw), p["output"]):
                    ok = False
                    break
            if ok:
                return lambda inp, a=fh, b=fw: downscale_grid(inp, a, b)
    return None

def try_gravity_strategy(train):
    for func in [gravity_down, gravity_up, gravity_left, gravity_right]:
        if all(grids_equal(func(p["input"], background_color(p["input"])), p["output"]) for p in train):
            return lambda inp, f=func: f(inp, background_color(inp))
    return None

def try_fill_enclosed_strategy(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    if all(grids_equal(fill_enclosed_regions(p["input"], background_color(p["input"])), p["output"]) for p in train):
        return lambda inp: fill_enclosed_regions(inp, background_color(inp))
    return None

def try_remove_border_strategy(train):
    for p in train:
        ih, iw = grid_dims(p["input"])
        oh, ow = grid_dims(p["output"])
        if oh != ih - 2 or ow != iw - 2:
            return None
    if all(grids_equal(remove_border(p["input"]), p["output"]) for p in train):
        return remove_border
    return None

def try_add_border_strategy(train):
    for p in train:
        ih, iw = grid_dims(p["input"])
        oh, ow = grid_dims(p["output"])
        if oh != ih + 2 or ow != iw + 2:
            return None
    bc = Counter(extract_border(train[0]["output"])).most_common(1)[0][0]
    if all(grids_equal(add_border(p["input"], bc), p["output"]) for p in train):
        return lambda inp, c=bc: add_border(inp, c)
    return None

def try_symmetry_completion_strategy(train):
    for func in [make_4way_symmetric, make_horizontally_symmetric,
                 make_vertically_symmetric, complete_symmetry_bg_aware]:
        if all(grids_equal(func(p["input"]), p["output"]) for p in train):
            return func
    return None

def try_replace_single_color_strategy(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    diffs = defaultdict(set)
    for p in train:
        inp, out = p["input"], p["output"]
        h, w = grid_dims(inp)
        for i in range(h):
            for j in range(w):
                if inp[i][j] != out[i][j]:
                    diffs[inp[i][j]].add(out[i][j])
    if len(diffs) == 1:
        old_c = list(diffs.keys())[0]
        targets = diffs[old_c]
        if len(targets) == 1:
            new_c = list(targets)[0]
            if all(grids_equal(replace_color(p["input"], old_c, new_c), p["output"]) for p in train):
                return lambda inp, o=old_c, n=new_c: replace_color(inp, o, n)
    return None

def try_largest_object_strategy(train):
    for p in train:
        bg = background_color(p["input"])
        objects = extract_multicolor_objects(p["input"], bg)
        if not objects:
            return None
        largest = max(objects, key=lambda o: o["size"])
        if not grids_equal(largest["grid"], p["output"]):
            return None
    def get_largest(inp):
        bg = background_color(inp)
        objects = extract_multicolor_objects(inp, bg)
        if not objects:
            return copy_grid(inp)
        return max(objects, key=lambda o: o["size"])["grid"]
    return get_largest

def try_smallest_object_strategy(train):
    for p in train:
        bg = background_color(p["input"])
        objects = extract_multicolor_objects(p["input"], bg)
        if not objects:
            return None
        smallest = min(objects, key=lambda o: o["size"])
        if not grids_equal(smallest["grid"], p["output"]):
            return None
    def get_smallest(inp):
        bg = background_color(inp)
        objects = extract_multicolor_objects(inp, bg)
        if not objects:
            return copy_grid(inp)
        return min(objects, key=lambda o: o["size"])["grid"]
    return get_smallest

def try_denoise_strategy(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    for threshold in [1, 2, 3]:
        def make_denoise(t):
            def denoise(inp):
                bg = background_color(inp)
                cc = color_counts(inp)
                h, w = grid_dims(inp)
                return [[bg if cc[inp[i][j]] <= t else inp[i][j] for j in range(w)] for i in range(h)]
            return denoise
        fn = make_denoise(threshold)
        if all(grids_equal(fn(p["input"]), p["output"]) for p in train):
            return fn
    return None

def try_mirror_and_extend_strategy(train):
    combos = [
        ("h_right", lambda x: concat_horizontal([x, flip_h(x)])),
        ("h_left", lambda x: concat_horizontal([flip_h(x), x])),
        ("v_down", lambda x: concat_vertical([x, flip_v(x)])),
        ("v_up", lambda x: concat_vertical([flip_v(x), x])),
        ("4way", lambda x: concat_vertical([
            concat_horizontal([x, flip_h(x)]),
            concat_horizontal([flip_v(x), rotate180(x)])
        ])),
        ("4way_alt", lambda x: concat_vertical([
            concat_horizontal([flip_h(x), x]),
            concat_horizontal([rotate180(x), flip_v(x)])
        ])),
    ]
    for name, func in combos:
        if all(grids_equal(func(p["input"]), p["output"]) for p in train):
            return func
    return None

def try_overlay_split_halves_strategy(train):
    split_fns = [
        ("h2", lambda g: split_grid_horizontal(g, 2)),
        ("v2", lambda g: split_grid_vertical(g, 2)),
        ("h3", lambda g: split_grid_horizontal(g, 3)),
        ("v3", lambda g: split_grid_vertical(g, 3)),
    ]
    merge_fns = [or_grids, and_grids, xor_grids]

    for sf_name, split_fn in split_fns:
        for merge_fn in merge_fns:
            for bg_val in [0]:
                try:
                    ok = True
                    for p in train:
                        parts = split_fn(p["input"])
                        if len(parts) < 2:
                            ok = False
                            break
                        # Check all parts same size
                        ph, pw = grid_dims(parts[0])
                        if not all(grid_dims(pt) == (ph, pw) for pt in parts):
                            ok = False
                            break
                        result = parts[0]
                        for pt in parts[1:]:
                            result = merge_fn(result, pt, bg_val)
                        if not grids_equal(result, p["output"]):
                            ok = False
                            break
                    if ok:
                        def make_solver(sf, mf, bv):
                            def solver(inp):
                                ps = sf(inp)
                                r = ps[0]
                                for pt in ps[1:]:
                                    r = mf(r, pt, bv)
                                return r
                            return solver
                        return make_solver(split_fn, merge_fn, bg_val)
                except Exception:
                    continue
    return None

def try_extract_repeating_unit_strategy(train):
    for p in train:
        unit = find_repeating_unit(p["input"])
        if not grids_equal(unit, p["output"]):
            return None
    return lambda inp: find_repeating_unit(inp)

def try_row_col_duplication_strategy(train):
    for n in range(2, 5):
        if all(grids_equal(duplicate_rows(p["input"], n), p["output"]) for p in train):
            return lambda inp, nn=n: duplicate_rows(inp, nn)
        if all(grids_equal(duplicate_cols(p["input"], n), p["output"]) for p in train):
            return lambda inp, nn=n: duplicate_cols(inp, nn)
    return None

def try_per_object_transform_strategy(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    for tname, tfn in ALL_RIGID_TRANSFORMS[1:]:  # skip identity
        ok = True
        for p in train:
            bg = background_color(p["input"])
            h, w = grid_dims(p["input"])
            objects = extract_multicolor_objects(p["input"], bg)
            result = [[bg] * w for _ in range(h)]
            for obj in objects:
                tobj = tfn(obj["grid"])
                th, tw = grid_dims(tobj)
                r1, c1, _, _ = obj["bbox"]
                for di in range(th):
                    for dj in range(tw):
                        ni, nj = r1 + di, c1 + dj
                        if 0 <= ni < h and 0 <= nj < w and tobj[di][dj] != bg:
                            result[ni][nj] = tobj[di][dj]
            if not grids_equal(result, p["output"]):
                ok = False
                break
        if ok:
            def make_solver(tf):
                def solver(inp):
                    bg = background_color(inp)
                    h, w = grid_dims(inp)
                    objs = extract_multicolor_objects(inp, bg)
                    res = [[bg] * w for _ in range(h)]
                    for obj in objs:
                        to = tf(obj["grid"])
                        th, tw = grid_dims(to)
                        r1, c1, _, _ = obj["bbox"]
                        for di in range(th):
                            for dj in range(tw):
                                ni, nj = r1 + di, c1 + dj
                                if 0 <= ni < h and 0 <= nj < w and to[di][dj] != bg:
                                    res[ni][nj] = to[di][dj]
                    return res
                return solver
            return make_solver(tfn)
    return None

def try_pixel_checker_rule(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    rules = {}
    ok = True
    for p in train:
        inp, out = p["input"], p["output"]
        h, w = grid_dims(inp)
        for i in range(h):
            for j in range(w):
                key = (inp[i][j], (i + j) % 2)
                if key in rules:
                    if rules[key] != out[i][j]:
                        ok = False
                        break
                else:
                    rules[key] = out[i][j]
            if not ok:
                break
        if not ok:
            break
    if ok and rules:
        def apply_rule(inp, r=dict(rules)):
            h, w = grid_dims(inp)
            return [[r.get((inp[i][j], (i + j) % 2), inp[i][j]) for j in range(w)] for i in range(h)]
        if all(grids_equal(apply_rule(p["input"]), p["output"]) for p in train):
            return apply_rule
    return None

def try_split_and_overlay_dividers(train):
    for p in train:
        h_div, v_div = detect_grid_divisions(p["input"])
        if not h_div and not v_div:
            return None

    merge_fns = [or_grids, and_grids, xor_grids]
    for merge_fn in merge_fns:
        ok = True
        for p in train:
            h_div, v_div = detect_grid_divisions(p["input"])
            cells = split_by_dividers(p["input"], h_div, v_div)
            flat = [c for row in cells for c in row]
            if len(flat) < 2:
                ok = False
                break
            ch, cw = grid_dims(flat[0])
            if not all(grid_dims(c) == (ch, cw) for c in flat):
                ok = False
                break
            result = flat[0]
            for fc in flat[1:]:
                result = merge_fn(result, fc, 0)
            if not grids_equal(result, p["output"]):
                ok = False
                break
        if ok:
            def make_solver(mf):
                def solver(inp):
                    hd, vd = detect_grid_divisions(inp)
                    cs = split_by_dividers(inp, hd, vd)
                    fc = [c for row in cs for c in row]
                    if len(fc) < 2:
                        return copy_grid(inp)
                    r = fc[0]
                    for c in fc[1:]:
                        r = mf(r, c, 0)
                    return r
                return solver
            return make_solver(merge_fn)
    return None

def try_subgrid_extraction(train):
    """Cek apakah output selalu subgrid dari input di posisi tertentu."""
    for p in train:
        inp, out = p["input"], p["output"]
        oh, ow = grid_dims(out)
        ih, iw = grid_dims(inp)
        found = False
        for r in range(ih - oh + 1):
            for c in range(iw - ow + 1):
                if grids_equal(crop(inp, r, c, r + oh, c + ow), out):
                    found = True
                    break
            if found:
                break
        if not found:
            return None

    # Try fixed positions
    for anchor in ["top_left", "top_right", "bottom_left", "bottom_right", "center"]:
        ok = True
        for p in train:
            inp, out = p["input"], p["output"]
            oh, ow = grid_dims(out)
            ih, iw = grid_dims(inp)
            if anchor == "top_left":
                r, c = 0, 0
            elif anchor == "top_right":
                r, c = 0, iw - ow
            elif anchor == "bottom_left":
                r, c = ih - oh, 0
            elif anchor == "bottom_right":
                r, c = ih - oh, iw - ow
            elif anchor == "center":
                r, c = (ih - oh) // 2, (iw - ow) // 2
            else:
                r, c = 0, 0
            if r < 0 or c < 0:
                ok = False
                break
            if not grids_equal(crop(inp, r, c, r + oh, c + ow), out):
                ok = False
                break
        if ok:
            def make_crop(anc):
                def solver(inp):
                    # Need output dims — infer from training
                    oh = train[0]["output"]
                    odims = grid_dims(oh)
                    ih, iw = grid_dims(inp)
                    ooh, oow = odims
                    if anc == "top_left":
                        return crop(inp, 0, 0, ooh, oow)
                    elif anc == "top_right":
                        return crop(inp, 0, iw - oow, ooh, iw)
                    elif anc == "bottom_left":
                        return crop(inp, ih - ooh, 0, ih, oow)
                    elif anc == "bottom_right":
                        return crop(inp, ih - ooh, iw - oow, ih, iw)
                    elif anc == "center":
                        return crop(inp, (ih - ooh) // 2, (iw - oow) // 2,
                                    (ih - ooh) // 2 + ooh, (iw - oow) // 2 + oow)
                    return copy_grid(inp)
                return solver
            # Check if output dims are consistent
            out_dims = set(grid_dims(p["output"]) for p in train)
            if len(out_dims) == 1:
                return make_crop(anchor)
    return None

def try_recolor_by_size(train):
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    size_to_color = {}
    for p in train:
        bg = background_color(p["input"])
        objs_in = extract_objects(p["input"], bg)
        objs_out = extract_objects(p["output"], bg)
        if len(objs_in) != len(objs_out):
            return None
        for oi in objs_in:
            for oo in objs_out:
                if oi["bbox"] == oo["bbox"] and oi["size"] == oo["size"]:
                    s = oi["size"]
                    if s in size_to_color and size_to_color[s] != oo["color"]:
                        return None
                    size_to_color[s] = oo["color"]
                    break
    if not size_to_color:
        return None

    def recolor(inp, s2c=dict(size_to_color)):
        bg = background_color(inp)
        result = copy_grid(inp)
        objs = extract_objects(inp, bg)
        for obj in objs:
            if obj["size"] in s2c:
                for r, c in obj["positions"]:
                    result[r][c] = s2c[obj["size"]]
        return result

    if all(grids_equal(recolor(p["input"]), p["output"]) for p in train):
        return recolor
    return None

def try_output_fixed_size_crop(train):
    """Output = crop at position of a special marker/color."""
    out_dims = set(grid_dims(p["output"]) for p in train)
    if len(out_dims) != 1:
        return None
    oh, ow = list(out_dims)[0]

    # Try: crop around each unique color's centroid
    for p in train:
        all_c = unique_colors(p["input"]) - {background_color(p["input"])}
        for color in all_c:
            positions = [(i, j) for i in range(len(p["input"])) for j in range(len(p["input"][0]))
                         if p["input"][i][j] == color]
            if not positions:
                continue
            cr = sum(r for r, c in positions) // len(positions)
            cc = sum(c for r, c in positions) // len(positions)
            r1 = max(0, cr - oh // 2)
            c1 = max(0, cc - ow // 2)
            ih, iw = grid_dims(p["input"])
            r1 = min(r1, ih - oh)
            c1 = min(c1, iw - ow)
            if r1 >= 0 and c1 >= 0 and r1 + oh <= ih and c1 + ow <= iw:
                if grids_equal(crop(p["input"], r1, c1, r1 + oh, c1 + ow), p["output"]):
                    # Verify all
                    def make_solver(col, ooh, oow):
                        def solver(inp):
                            pos = [(i, j) for i in range(len(inp)) for j in range(len(inp[0]))
                                   if inp[i][j] == col]
                            if not pos:
                                return crop(inp, 0, 0, ooh, oow)
                            cr2 = sum(r for r, c in pos) // len(pos)
                            cc2 = sum(c for r, c in pos) // len(pos)
                            ih2, iw2 = grid_dims(inp)
                            r12 = max(0, min(cr2 - ooh // 2, ih2 - ooh))
                            c12 = max(0, min(cc2 - oow // 2, iw2 - oow))
                            return crop(inp, r12, c12, r12 + ooh, c12 + oow)
                        return solver
                    solver = make_solver(color, oh, ow)
                    if all(grids_equal(solver(pp["input"]), pp["output"]) for pp in train):
                        return solver
    return None

def try_scale_then_color_map(train):
    """Scale + color map."""
    for fh in range(2, 5):
        for fw in range(2, 5):
            ok = True
            for p in train:
                if len(p["output"]) != len(p["input"]) * fh or len(p["output"][0]) != len(p["input"][0]) * fw:
                    ok = False
                    break
            if not ok:
                continue
            scaled_train = []
            for p in train:
                scaled_train.append({"input": scale_grid(p["input"], fh, fw), "output": p["output"]})
            cmap = infer_color_map(scaled_train)
            if all(grids_equal(color_map(scale_grid(p["input"], fh, fw), cmap), p["output"]) for p in train):
                return lambda inp, a=fh, b=fw, m=cmap: color_map(scale_grid(inp, a, b), m)
    return None

def try_tile_then_color_map(train):
    for rh in range(1, 5):
        for rw in range(1, 5):
            if rh == 1 and rw == 1:
                continue
            ok = True
            for p in train:
                if len(p["output"]) != len(p["input"]) * rh or len(p["output"][0]) != len(p["input"][0]) * rw:
                    ok = False
                    break
            if not ok:
                continue
            tiled_train = []
            for p in train:
                tiled_train.append({"input": tile_grid(p["input"], rh, rw), "output": p["output"]})
            cmap = infer_color_map(tiled_train)
            if all(grids_equal(color_map(tile_grid(p["input"], rh, rw), cmap), p["output"]) for p in train):
                return lambda inp, a=rh, b=rw, m=cmap: color_map(tile_grid(inp, a, b), m)
    return None

def try_crop_then_transform(train):
    """Crop to content, then apply a rigid transform."""
    for p in train:
        bg = background_color(p["input"])
        cropped = crop_to_content(p["input"], bg)
        for tname, tfn in ALL_RIGID_TRANSFORMS:
            if grids_equal(tfn(cropped), p["output"]):
                # Check all
                def make_solver(tf):
                    def solver(inp):
                        b = background_color(inp)
                        return tf(crop_to_content(inp, b))
                    return solver
                solver = make_solver(tfn)
                if all(grids_equal(solver(pp["input"]), pp["output"]) for pp in train):
                    return solver
    return None

def try_fill_by_neighbor_rule(train):
    """Cek apakah setiap cell bg diisi berdasarkan neighbor terdekat."""
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    # Check if only bg cells change
    for p in train:
        bg = background_color(p["input"])
        inp, out = p["input"], p["output"]
        h, w = grid_dims(inp)
        for i in range(h):
            for j in range(w):
                if inp[i][j] != bg and inp[i][j] != out[i][j]:
                    return None  # Non-bg cells also change → not this pattern
    # Try: fill each bg cell with the color of the nearest non-bg cell
    def fill_nearest(inp):
        bg = background_color(inp)
        h, w = grid_dims(inp)
        result = copy_grid(inp)
        non_bg = [(i, j, inp[i][j]) for i in range(h) for j in range(w) if inp[i][j] != bg]
        if not non_bg:
            return result
        for i in range(h):
            for j in range(w):
                if inp[i][j] == bg:
                    min_dist = float('inf')
                    best_color = bg
                    for ni, nj, nc in non_bg:
                        d = abs(ni - i) + abs(nj - j)
                        if d < min_dist:
                            min_dist = d
                            best_color = nc
                    result[i][j] = best_color
        return result

    if all(grids_equal(fill_nearest(p["input"]), p["output"]) for p in train):
        return fill_nearest

    # Try: fill row-wise (extend colors horizontally)
    def fill_row_extend(inp):
        bg = background_color(inp)
        h, w = grid_dims(inp)
        result = copy_grid(inp)
        for i in range(h):
            last_color = bg
            for j in range(w):
                if result[i][j] != bg:
                    last_color = result[i][j]
                elif last_color != bg:
                    result[i][j] = last_color
            last_color = bg
            for j in range(w - 1, -1, -1):
                if inp[i][j] != bg:
                    last_color = inp[i][j]
                elif last_color != bg and result[i][j] == bg:
                    result[i][j] = last_color
        return result

    if all(grids_equal(fill_row_extend(p["input"]), p["output"]) for p in train):
        return fill_row_extend

    # Try: fill column-wise
    def fill_col_extend(inp):
        bg = background_color(inp)
        h, w = grid_dims(inp)
        result = copy_grid(inp)
        for j in range(w):
            last_color = bg
            for i in range(h):
                if result[i][j] != bg:
                    last_color = result[i][j]
                elif last_color != bg:
                    result[i][j] = last_color
            last_color = bg
            for i in range(h - 1, -1, -1):
                if inp[i][j] != bg:
                    last_color = inp[i][j]
                elif last_color != bg and result[i][j] == bg:
                    result[i][j] = last_color
        return result

    if all(grids_equal(fill_col_extend(p["input"]), p["output"]) for p in train):
        return fill_col_extend

    return None

def try_color_swap(train):
    """Cek apakah dua warna di-swap."""
    for p in train:
        if grid_dims(p["input"]) != grid_dims(p["output"]):
            return None
    inp_colors = unique_colors(train[0]["input"])
    for c1 in inp_colors:
        for c2 in inp_colors:
            if c1 >= c2:
                continue
            mapping = {c1: c2, c2: c1}
            if all(grids_equal(color_map(p["input"], mapping), p["output"]) for p in train):
                return lambda inp, m=dict(mapping): color_map(inp, m)
    return None

def try_output_count_grid(train):
    """Output = small grid encoding counts of colors or objects."""
    for p in train:
        oh, ow = grid_dims(p["output"])
        if oh > 10 or ow > 10:
            return None
    # Try: output is 1x1 with count of non-bg cells
    all_1x1 = all(grid_dims(p["output"]) == (1, 1) for p in train)
    if all_1x1:
        for p in train:
            bg = background_color(p["input"])
            count = sum(1 for row in p["input"] for c in row if c != bg)
            if p["output"][0][0] != count:
                break
        else:
            def count_solver(inp):
                bg = background_color(inp)
                count = sum(1 for row in inp for c in row if c != bg)
                return [[count]]
            return count_solver

    # Try: count objects
    if all_1x1:
        for p in train:
            bg = background_color(p["input"])
            objs = extract_multicolor_objects(p["input"], bg)
            if p["output"][0][0] != len(objs):
                break
        else:
            def obj_count_solver(inp):
                bg = background_color(inp)
                return [[len(extract_multicolor_objects(inp, bg))]]
            return obj_count_solver

    return None

def try_majority_per_row_or_col(train):
    """Output = majority color per row or column."""
    for p in train:
        oh, ow = grid_dims(p["output"])
        ih, iw = grid_dims(p["input"])
        # Column output: 1 x iw
        if oh == 1 and ow == iw:
            result = [[Counter([p["input"][i][j] for i in range(ih)]).most_common(1)[0][0] for j in range(iw)]]
            if grids_equal(result, p["output"]):
                continue
            else:
                break
    else:
        if all(grid_dims(p["output"]) == (1, grid_dims(p["input"])[1]) for p in train):
            def col_majority(inp):
                ih, iw = grid_dims(inp)
                return [[Counter([inp[i][j] for i in range(ih)]).most_common(1)[0][0] for j in range(iw)]]
            return col_majority

    # Row majority: ih x 1
    for p in train:
        oh, ow = grid_dims(p["output"])
        ih, iw = grid_dims(p["input"])
        if ow == 1 and oh == ih:
            result = [[Counter(p["input"][i]).most_common(1)[0][0]] for i in range(ih)]
            if not grids_equal(result, p["output"]):
                break
    else:
        if all(grid_dims(p["output"]) == (grid_dims(p["input"])[0], 1) for p in train):
            def row_majority(inp):
                ih, iw = grid_dims(inp)
                return [[Counter(inp[i]).most_common(1)[0][0]] for i in range(ih)]
            return row_majority

    return None

# ============================================================================
# 16. MAIN SOLVER FUNCTION
# ============================================================================

def find_program(task):
    train = task["train"]
    if not train:
        return None
    strategies = [
        try_output_constant,
        try_rigid_transforms,
        try_color_map_only,
        try_color_swap,
        try_replace_single_color_strategy,
        try_crop_to_content_strategy,
        try_crop_to_specific_color,
        try_scale,
        try_tile,
        try_downscale,
        try_gravity_strategy,
        try_fill_enclosed_strategy,
        try_remove_border_strategy,
        try_add_border_strategy,
        try_symmetry_completion_strategy,
        try_largest_object_strategy,
        try_smallest_object_strategy,
        try_denoise_strategy,
        try_mirror_and_extend_strategy,
        try_overlay_split_halves_strategy,
        try_extract_repeating_unit_strategy,
        try_row_col_duplication_strategy,
        try_per_object_transform_strategy,
        try_pixel_checker_rule,
        try_split_and_overlay_dividers,
        try_transform_then_color_map,
        try_crop_then_transform,
        try_fill_by_neighbor_rule,
        try_output_count_grid,
        try_majority_per_row_or_col,
        try_subgrid_extraction,
        try_output_fixed_size_crop,
        try_recolor_by_size,
        try_scale_then_color_map,
        try_tile_then_color_map,
    ]
    for strat in strategies:
        try:
            prog = strat(train)
            if prog is not None:
                return prog
        except Exception:
            continue
    return None

def safe_fallback_prediction(inp):
    h, w = grid_dims(inp)
    bg = background_color(inp)
    pred1 = copy_grid(inp)
    pred2 = [[bg] * w for _ in range(h)]
    return pred1, pred2

def solve_task(task):
    results = []
    program = find_program(task)
    for test_case in task["test"]:
        inp = test_case["input"]
        if program is not None:
            try:
                pred1 = program(inp)
            except Exception:
                pred1 = copy_grid(inp)
        else:
            pred1 = copy_grid(inp)
        _, pred2 = safe_fallback_prediction(inp)
        results.append({
            "attempt_1": pred1,
            "attempt_2": pred2
        })
    return results

# ============================================================================
# 17. GENERATE SUBMISSION.JSON
# ============================================================================

# Generate submission
submission = {}

print(f"Processing {len(data)} tasks...")
for idx, (task_id, task) in enumerate(data.items()):
    if idx % 10 == 0:
        print(f"Progress: {idx}/{len(data)} tasks")
    submission[task_id] = solve_task(task)

# Write to file
with open("submission.json", "w") as f:
    json.dump(submission, f)

print("✅ submission.json created successfully!")
print(f"📊 Total tasks processed: {len(data)}")

Processing 120 tasks...
Progress: 0/120 tasks
Progress: 10/120 tasks
Progress: 20/120 tasks
Progress: 30/120 tasks
Progress: 40/120 tasks
Progress: 50/120 tasks
Progress: 60/120 tasks
Progress: 70/120 tasks
Progress: 80/120 tasks
Progress: 90/120 tasks
Progress: 100/120 tasks
Progress: 110/120 tasks
✅ submission.json created successfully!
📊 Total tasks processed: 120
